<a href="https://www.kaggle.com/code/shivams811/credit-card-churn-ann-stratifiedkfold?scriptVersionId=272037972" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# 💳 Credit Card Churn Prediction | ANN + Stratified KFold 

### 📘 Overview
This notebook predicts **customer churn** for a bank's credit card users using an **Artificial Neural Network (ANN)**.  
The goal is to identify customers likely to leave the bank based on demographic and financial attributes.

### 🧩 Dataset
<!-- - **Source:** Kaggle - [Customer Churn Dataset](https://www.kaggle.com/) -->
- **Rows:** 10,000  
- **Target Variable:** `Exited` (1 = Churned, 0 = Active)  
- **Features:** CreditScore, Geography, Gender, Age, Balance, NumOfProducts, etc.

### ⚙️ Approach
- Data preprocessing (encoding categorical features, scaling)
- Built a deep **Artificial Neural Network (ANN)** using TensorFlow/Keras
- Applied **Stratified K-Fold Cross Validation (k=10)** for robust model evaluation
- Used **EarlyStopping** to prevent overfitting

### 🧠 Model Architecture
- Dense(64, ReLU) → Dropout(0.3)  
- Dense(32, ReLU)  
- Dense(16, ReLU) → Dropout(0.2)  
- Dense(8, ReLU)  
- Dense(1, Sigmoid)

### 📈 Results
- **Average CV Accuracy:** `0.86099`
- **Loss Function:** Binary Crossentropy  
- **Optimizer:** Adam  
- **Evaluation Metric:** Accuracy
- **Final Model Accuracy:** 0.8565 

### 🚀 Key Insights
<!-- - Customers with **low balance, high age**, and **low activity** are more likely to churn. -->
- The ANN model generalizes well with consistent cross-validation performance.

---




## Importing Libraries

In [ ]:
import pandas as pd
import numpy as np
import tensorflow
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold
from tensorflow import keras
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import accuracy_score, roc_curve, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

import random
import os
import tensorflow as tf


os.environ['PYTHONHASHSEED'] = '42'
os.environ['TF_DETERMINISTIC_OPS'] = '1'

random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

## Loading Data

In [ ]:
df = pd.read_csv('/kaggle/input/credit-card-customer-churn-prediction/Churn_Modelling.csv')

## EDA

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df['Exited'].value_counts()

In [ ]:
df['Geography'].value_counts()

## Dropping Useless Columns

In [ ]:
df.drop(columns=['RowNumber', 'CustomerId', 'Surname'], inplace =True)

In [ ]:
df.head()

## Feature Encoding

In [ ]:
df = pd.get_dummies(df, columns=['Geography', 'Gender'], drop_first=True)

In [ ]:
df.head()

In [ ]:
X = df.drop(columns=['Exited'])
y = df['Exited']

## Cross Validation

In [ ]:
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

fold_accuracies = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"\n----- Fold {fold+1} -----")
    
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val = scaler.transform(X_val)
    
    
    model = Sequential([
        Dense(64, activation='relu', input_dim=11),
        Dropout(0.3),
        Dense(32, activation='relu'),
        Dense(16, activation='relu'),
         Dropout(0.2),
        Dense(8, activation='relu'),
        Dense(1, activation='sigmoid')
    ])
    
    model.compile(optimizer='Adam',
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
    
   
    es = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
    
    model.fit(X_train, y_train,
              validation_data=(X_val, y_val),
              epochs=50,
              batch_size=32,
              callbacks=[es],
              verbose=0)
    
    _, val_acc = model.evaluate(X_val, y_val, verbose=0)
    print(f"Fold {fold+1} Validation Accuracy: {val_acc:.4f}")
    
    fold_accuracies.append(val_acc)

print("\nAverage CV Accuracy:", np.mean(fold_accuracies))

## Splitting Data

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2, stratify=y)

## Scaling Data

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

## Model Building

In [ ]:
final_model = Sequential([
    Dense(64, activation='relu', input_dim=11),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(16, activation='relu'),
    Dropout(0.2),
    Dense(8, activation='relu'),
    Dense(1, activation='sigmoid')
])

final_model.compile(optimizer='Adam',
                    loss='binary_crossentropy',
                    metrics=['accuracy'])

es = EarlyStopping(monitor='loss', patience=5, restore_best_weights=True)

final_model.fit(X_train, y_train, epochs=50, batch_size=32, callbacks=[es], verbose=1)


## Model Evaluation

In [ ]:
model.summary()

In [ ]:
y_pred_prob = final_model.predict(X_test)

### Here I have randomly taken 0.5 as it is general, to find more accurate threshold use roc-auc curve.

In [ ]:
y_pred = (y_pred_prob >= 0.5).astype(int)

In [ ]:
print("Model Accuracy: ", accuracy_score(y_test, y_pred))